# 🔬 Pulp Fibril Segmentation — Google Colab Training (~90 min)

### ⚡ FAST VERSION — Optimized for Colab Free T4 GPU

**What this notebook does (in order):**
1. Installs all Python libraries
2. Mounts your Google Drive (to save checkpoints persistently)
3. Clones your project from GitHub OR uploads from local zip
4. Generates 250 synthetic microscopy images
5. Trains the model for 10 epochs (~90 min on T4)
6. Saves `best_checkpoint.pth` directly to your Google Drive
7. Shows a demo inference on 3 test images

**Estimated time:** ~10 min setup + ~90 min training = ~100 min total

---
> ⚠️ **Before starting:** Go to `Runtime → Change Runtime Type → GPU → T4 GPU → Save`

---
## 🔧 CELL 1 — Verify GPU
**What it does:** Checks that you are connected to a GPU. If it says `No GPU` — go to Runtime → Change Runtime Type → GPU.

In [ ]:
import subprocess, torch

print('='*50)
print('GPU CHECK')
print('='*50)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU:  {name}')
    print(f'✅ VRAM: {vram:.1f} GB')
    print(f'✅ PyTorch: {torch.__version__}')
    print('\n🟢 Ready to train!')
else:
    print('❌ NO GPU DETECTED!')
    print('Go to: Runtime → Change Runtime Type → GPU → T4 GPU → Save')
    print('Then restart this notebook and run again.')
    raise RuntimeError('GPU required — please enable it first!')

DEVICE = 'cuda'

---
## 💾 CELL 2 — Mount Google Drive
**What it does:** Connects your Google Drive so we can save the trained model checkpoint there. This is important because Colab sessions reset after ~12-24hrs — your Google Drive persists forever.

When you run this cell:
- A popup appears asking for permission → click **Allow** → copy the code → paste it in the box

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/fibril_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

print(f'✅ Google Drive mounted!')
print(f'✅ Checkpoint will be saved to: {DRIVE_CHECKPOINT_DIR}')
print(f'   (This folder is in YOUR Google Drive → stays safe even if Colab disconnects!)')

---
## 📦 CELL 3 — Install Python Libraries
**What it does:** Installs all packages the project needs.

- `timm` — for Swin Transformer pretrained weights
- `albumentations` — for image augmentation
- `scikit-image` — for skeleton extraction
- `networkx` — for fiber graph analysis
- `einops` — for tensor reshaping utilities

**Expected time:** ~3-4 minutes

In [ ]:
print('Installing packages...')
packages = [
    'timm>=0.9.12',
    'albumentations>=1.3.1',
    'scikit-image>=0.21.0',
    'networkx>=3.1',
    'einops>=0.7.0',
    'scipy>=1.11.0',
    'opencv-python-headless',
]

for pkg in packages:
    result = subprocess.run(
        ['pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    print(f'  ✅ {pkg.split(">")[0]}')

print('\n✅ All packages installed!')

---
## 📁 CELL 4 — Upload Project Files
**What it does:** Uploads your project from your laptop to Colab.

**You have 2 choices — pick ONE:**

**Option A (Recommended) — Upload a ZIP:**  
1. On your laptop, zip the entire `DL CP` folder: right-click → Send to → Compressed folder
2. Run Option A cells below — a file browser will pop up
3. Select your `DL_CP.zip` file — it uploads automatically

**Option B — GitHub (if you push it there):**  
Replace `YOUR_REPO_URL` with your GitHub repository URL

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OPTION A: Upload ZIP from your laptop
# ═══════════════════════════════════════════════════════════════════
# Run this cell. A file picker will pop up. Select your DL_CP.zip

from google.colab import files
import zipfile, os, sys

print('A file picker will appear below → Select your DL_CP.zip file')
uploaded = files.upload()  # <-- opens file picker

# Unzip it
for filename in uploaded:
    if filename.endswith('.zip'):
        print(f'Unzipping {filename}...')
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('/content/')
        print('✅ Unzipped!')

# ── Find the project root (wherever synthetic_gen.py lives)
import glob
gen_files = glob.glob('/content/**/synthetic_gen.py', recursive=True)
if gen_files:
    PROJECT_ROOT = str(Path(gen_files[0]).parent.parent)
    print(f'✅ Project found at: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '/content/DL CP'  # adjust if needed
    print(f'⚠️  Using default path: {PROJECT_ROOT}')

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OPTION B: Clone from GitHub (only if you pushed your code there)
# ═══════════════════════════════════════════════════════════════════
# SKIP THIS CELL if you used Option A above!

# Replace with your GitHub repo URL:
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {REPO_URL} /content/project

PROJECT_ROOT = '/content/project'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'✅ Cloned from GitHub → {PROJECT_ROOT}')

---
## 🧬 CELL 5 — Generate Synthetic Dataset
**What it does:** Runs your `generate_dataset.py` script to create 250 fake microscopy images with perfect ground-truth fiber masks.

Each image is generated procedurally:
- Random bezier curves = fiber body
- Hair-like strands = fibrils
- Alpha blending = translucency effect
- Gaussian noise + blur = real microscope simulation

**Expected time:** ~8-12 minutes on Colab CPU

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/fibril_data/data/synthetic')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

annot_path = DATA_ROOT / 'annotations.json'

if annot_path.exists():
    import json
    with open(annot_path) as f:
        coco_info = json.load(f)
    print(f'✅ Dataset already exists: {len(coco_info["images"])} images')
else:
    print('Generating 250 synthetic fibril images...')
    print('(This takes ~10 minutes — go grab a coffee ☕)')
    !cd {PROJECT_ROOT} && python data_pipeline/generate_dataset.py \
        --n 250 \
        --output {DATA_ROOT.parent.parent} \
        --size 256 \
        --seed 42

    if annot_path.exists():
        print('\n✅ Dataset generated successfully!')
    else:
        print('❌ Generation failed — check error above')

---
## 🖼️ CELL 6 — Preview Sample Images
**What it does:** Shows you 3 random generated images with their fiber instance masks overlaid in different colors.

This is how you verify the data looks correct before training. Check:
- Each fiber should be a different color
- Fibers should look curved, not straight
- There should be visible hair-like strands off the main fiber body

In [ ]:
import cv2, json, random, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

with open(DATA_ROOT / 'annotations.json') as f:
    coco = json.load(f)

samples = random.sample(coco['images'], 3)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('FibrilSynth — Generated Training Data', fontsize=14, color='white')
fig.patch.set_facecolor('#1a1a2e')

for col, info in enumerate(samples):
    img = cv2.imread(str(DATA_ROOT / 'images' / info['file_name']), cv2.IMREAD_GRAYSCALE)
    masks_dir = DATA_ROOT / 'masks' / f"{info['id']:05d}"
    masks = [cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
             for mf in sorted(masks_dir.iterdir()) if mf.suffix == '.png']

    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR).astype(np.float32)
    colors = [(random.randint(80,255), random.randint(80,255), random.randint(80,255)) for _ in masks]
    for mask, color in zip(masks, colors):
        if mask is not None:
            colored = np.zeros_like(vis); colored[mask>127] = color
            vis = cv2.addWeighted(vis, 1.0, colored, 0.5, 0)

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'{info["file_name"]}\nOriginal', color='white', fontsize=9)
    axes[0, col].axis('off'); axes[0,col].set_facecolor('#1a1a2e')

    axes[1, col].imshow(vis.astype(np.uint8)[...,::-1])
    axes[1, col].set_title(f'{len(masks)} fiber instances', color='white', fontsize=9)
    axes[1, col].axis('off'); axes[1,col].set_facecolor('#1a1a2e')

plt.tight_layout()
plt.savefig('/content/sample_preview.png', dpi=120, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'\n✅ Looks good! Total images: {len(coco["images"])} | Total annotations: {len(coco["annotations"])}')

---
## 🧠 CELL 7 — Build and Test Model
**What it does:** Creates the Mask2Former model and runs a quick forward pass with a fake image to make sure everything is working.

Expected output:
- `pred_logits: torch.Size([1, 25, 2])` — 25 queries, each predicting class scores
- `pred_masks: torch.Size([1, 25, 64, 64])` — 25 binary mask predictions

If you see these shapes, the model is loaded correctly.

In [ ]:
import sys, os
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from models.mask2former import build_model
import torch

COLAB_CONFIG = {
    'backbone':   'swin_tiny',
    'pretrained': True,     # Downloads ImageNet weights (~350MB, ~2min)
    'num_queries': 25,      # Fast: 25 queries instead of 50
    'hidden_dim': 256,
    'nheads': 8,
    'dec_layers': 3,        # Fast: 3 decoder layers instead of 6
    'num_classes': 1,
}

print('Building model (downloading Swin-T ImageNet weights ~350MB)...')
model = build_model(COLAB_CONFIG, device=DEVICE)

print('\nRunning test forward pass...')
dummy_input = torch.zeros(1, 1, 256, 256).to(DEVICE)  # Fake 256×256 grayscale image
with torch.no_grad():
    out = model(dummy_input)

print(f'\n✅ Forward pass successful!')
print(f'   pred_logits shape: {out["pred_logits"].shape}  → (batch, queries, classes)')
print(f'   pred_masks  shape: {out["pred_masks"].shape}   → (batch, queries, H/4, W/4)')
print(f'\n🟢 Model ready for training!')

del model, dummy_input, out  # Free VRAM before training
torch.cuda.empty_cache()

---
## 🚀 CELL 8 — TRAIN THE MODEL
**What it does:** Runs the full training loop.

**Training details:**
- **Dataset:** 250 images (175 train / 37 val / 38 test)
- **Epochs:** 10 (Phase 1: freeze backbone 2 epochs → Phase 2: joint training 8 epochs)
- **Losses:** Focal Loss (class) + Dice Loss (mask) + Binary Focal Loss (pixel-level)
- **Optimizer:** AdamW with cosine learning rate schedule
- **AMP:** Mixed precision (float16) enabled → saves VRAM, trains faster
- **Checkpoint:** Saved to your Google Drive after every improvement

**What you'll see printed each epoch:**
```
Epoch  1/10 | Train: 12.34 | Val: 11.89 ✅ BEST | LR=5e-05/3e-04 | 487s
```
- `Train/Val` = loss values (lower = better)
- `✅ BEST` = this checkpoint was saved to Drive

**Expected time:** ~9 min/epoch × 10 epochs = **~90 minutes total**

In [ ]:
from training.train import train

TRAINING_CONFIG = {
    # ── Data settings ─────────────────────────────────────────────
    'data_root':     str(DATA_ROOT),
    'image_size':    256,         # 256×256 pixels per image
    'batch_size':    2,           # Process 2 images at a time on GPU
    'accum_steps':   4,           # Accumulate gradients: effective batch = 2×4 = 8
    'num_workers':   2,           # Background CPU workers for data loading

    # ── Training schedule ──────────────────────────────────────────
    'epochs':        10,          # Total training epochs
    'warmup_epochs': 2,           # First 2 epochs: backbone frozen → only decoder trains

    # ── Model architecture ─────────────────────────────────────────
    'backbone':      'swin_tiny', # 28M parameter transformer
    'pretrained':    True,        # Use ImageNet pretrained weights (faster convergence)
    'num_queries':   25,          # Max 25 fibers per image
    'hidden_dim':    256,         # Embedding size throughout transformer
    'nheads':        8,           # Attention heads in transformer
    'dec_layers':    3,           # Transformer decoder depth
    'num_classes':   1,           # 1 class: fibril

    # ── Optimizer ──────────────────────────────────────────────────
    'lr_backbone':   5e-5,        # Backbone LR (lower = don't destroy pretrained weights)
    'lr_head':       3e-4,        # Decoder LR (higher = learn fast from scratch)
    'weight_decay':  0.05,        # L2 regularization
    'grad_clip':     0.1,         # Cap gradient norm to prevent exploding gradients

    # ── Loss weights ───────────────────────────────────────────────
    'loss_weights': {'loss_ce': 2.0, 'loss_mask': 5.0, 'loss_dice': 5.0},

    # ── Output ─────────────────────────────────────────────────────
    'checkpoint_dir': DRIVE_CHECKPOINT_DIR,  # Save to Google Drive!
    'use_wandb':      False,
    'device':         DEVICE,
    'mixed_precision': True,      # fp16 training — mandatory on T4
    'seed':           42,
}

print('Starting training...')
print(f'Checkpoint dir: {DRIVE_CHECKPOINT_DIR}')
print(f'Estimated time: ~90 minutes\n')

train(TRAINING_CONFIG)
print('\n🎉 Training complete! Checkpoint saved to Google Drive.')

---
## 📊 CELL 9 — Plot Training Curves
**What it does:** Shows you a graph of train loss vs. validation loss over all 10 epochs.

**What to look for:**
- Both lines should go DOWN over time (losses decreasing = model learning)
- Val loss shouldn't go much higher than train loss (would mean overfitting)
- Good result: val loss drops from ~15+ down to ~4-6 by epoch 10

In [ ]:
import json, matplotlib.pyplot as plt

history_path = Path(DRIVE_CHECKPOINT_DIR) / 'training_history.json'
with open(history_path) as f:
    history = json.load(f)

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')

epochs = range(1, len(history['train_loss']) + 1)
ax.plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#4ecdc4', linewidth=2, markersize=5)
ax.plot(epochs, history['val_loss'],   's-', label='Val Loss',   color='#ff6b6b', linewidth=2, markersize=5)

best_ep = history['val_loss'].index(min(history['val_loss'])) + 1
ax.axvline(best_ep, color='#ffd700', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_ep})')

ax.set_xlabel('Epoch', color='white'); ax.set_ylabel('Loss', color='white')
ax.set_title('Training Progress — Pulp Fibril Segmentation', color='white', fontsize=13)
ax.tick_params(colors='white'); ax.legend(facecolor='#2a2a4a', labelcolor='white')
ax.grid(alpha=0.2, color='gray')
[s.set_edgecolor('gray') for s in ax.spines.values()]

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print(f'Best val loss: {min(history["val_loss"]):.4f} at epoch {best_ep}')
print('✅ Training curve saved to /content/training_curves.png')

---
## 🔍 CELL 10 — Run Inference Demo
**What it does:** Loads your best trained model and runs it on 3 test images.

For each image, you will see:
- **Left:** Original grayscale microscopy image
- **Right:** Color-coded segmentation — each detected fiber = different color

Below each image, the measured metrics are printed:
- `len` = fiber length in micrometers (μm)
- `tort` = tortuosity (how curved the fiber is, 1.0 = straight)
- `branch` = number of branching points
- `FI` = Fibrillation Index (higher = more fibrillated = better paper bonding)

In [ ]:
import random
from inference.predict import predict
from pathlib import Path
import matplotlib.pyplot as plt, cv2

BEST_CKPT = Path(DRIVE_CHECKPOINT_DIR) / 'best_checkpoint.pth'

if not BEST_CKPT.exists():
    print(f'❌ Checkpoint not found: {BEST_CKPT}')
    print('Did training complete? Check Cell 8 output.')
else:
    print(f'✅ Checkpoint found: {BEST_CKPT}')

    img_dir = DATA_ROOT / 'images'
    all_imgs = list(img_dir.glob('*.png'))
    test_imgs = random.sample(all_imgs, min(3, len(all_imgs)))

    for img_path in test_imgs:
        print(f'\nProcessing: {img_path.name}')

        metrics = predict(
            image_path=str(img_path),
            checkpoint_path=str(BEST_CKPT),
            output_dir='/content/predictions/',
            px_to_um=0.25,
            use_sr=False,       # Skip SR for speed (enable with use_sr=True if you want)
            device=DEVICE,
        )

        # Show the visualization
        vis_path = f'/content/predictions/{img_path.stem}_segmented.png'
        if Path(vis_path).exists():
            vis = cv2.imread(vis_path)
            plt.figure(figsize=(12, 5))
            plt.imshow(vis[...,::-1])
            plt.title(f'{img_path.name} — {len(metrics)} fibrils | '
                      f'Avg FI={sum(m["fibrillation_index"] for m in metrics)/max(len(metrics),1):.2f}',
                      color='white')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

---
## ⬇️ CELL 11 — Download Checkpoint to Your Laptop
**What it does:** Helps you download the trained model checkpoint.

**Option A (Recommended):** Your checkpoint is already in **Google Drive** at:
`My Drive → fibril_checkpoints → best_checkpoint.pth`
→ Just go to drive.google.com and download it from there.

**Option B:** Download directly from this Colab session:

In [ ]:
from google.colab import files
from pathlib import Path
import os

BEST_CKPT = Path(DRIVE_CHECKPOINT_DIR) / 'best_checkpoint.pth'

if BEST_CKPT.exists():
    size_mb = os.path.getsize(BEST_CKPT) / 1e6
    print(f'✅ Checkpoint ready: {size_mb:.1f} MB')
    print()
    print('OPTION A (Google Drive) — Already saved!')
    print('  → Go to: drive.google.com → fibril_checkpoints → best_checkpoint.pth → Download')
    print()
    print('OPTION B (Direct download from Colab):')
    files.download(str(BEST_CKPT))  # <-- This triggers a browser download
else:
    print('❌ Checkpoint not found — make sure training completed in Cell 8')